In [1]:
%%info


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1429,application_1765289937462_1417,pyspark,idle,Link,Link,None,
1436,application_1765289937462_1424,pyspark,idle,Link,Link,None,
1437,application_1765289937462_1425,pyspark,idle,Link,Link,None,
1438,application_1765289937462_1426,pyspark,idle,Link,Link,None,
1441,application_1765289937462_1429,pyspark,idle,Link,Link,None,


In [2]:
%%configure -f
{
    "conf":{
        "spark.executor.instances": "2",
        "spark.executor.memory": "2g",
        "spark.executor.cores": "1"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1429,application_1765289937462_1417,pyspark,idle,Link,Link,None,
1436,application_1765289937462_1424,pyspark,idle,Link,Link,None,
1437,application_1765289937462_1425,pyspark,idle,Link,Link,None,
1438,application_1765289937462_1426,pyspark,idle,Link,Link,None,
1441,application_1765289937462_1429,pyspark,idle,Link,Link,None,


In [3]:
from sedona.spark import *
from pyspark.sql.functions import col
from pyspark.sql import SparkSession

# Create spark Session
spark = SparkSession.builder \
    .appName("GeoJSON read") \
    .getOrCreate()

# Create sedona context
sedona = SedonaContext.create(spark)
# Read the file from s3
geojson_path = "s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Census_Blocks_2020.geojson"
blocks_df = sedona.read.format("geojson") \
            .option("multiLine", "true").load(geojson_path) \
            .selectExpr("explode(features) as features") \
            .select("features.*")
# Formatting magic
flattened_df = blocks_df.select( \
                [col(f"properties.{col_name}").alias(col_name) for col_name in \
                blocks_df.schema["properties"].dataType.fieldNames()] + ["geometry"]) \
            .drop("properties") \
            .drop("type")
# Print schema
flattened_df.printSchema()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1445,application_1765289937462_1433,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- BG20: string (nullable = true)
 |-- BG20FIP_CURRENT: string (nullable = true)
 |-- BGFIP20: string (nullable = true)
 |-- CB20: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- CITYCOMM: string (nullable = true)
 |-- CITYCOMM_CURRENT: string (nullable = true)
 |-- CITY_CURRENT: string (nullable = true)
 |-- COMM: string (nullable = true)
 |-- COMM_CURRENT: string (nullable = true)
 |-- COUNTY: string (nullable = true)
 |-- CT20: string (nullable = true)
 |-- CTCB20: string (nullable = true)
 |-- FEAT_TYPE: string (nullable = true)
 |-- FIP20: string (nullable = true)
 |-- FIP_CURRENT: string (nullable = true)
 |-- HD22: long (nullable = true)
 |-- HD_NAME: string (nullable = true)
 |-- HOUSING20: long (nullable = true)
 |-- OBJECTID: long (nullable = true)
 |-- POP20: long (nullable = true)
 |-- SPA22: long (nullable = true)
 |-- SPA_NAME: string (nullable = true)
 |-- SUP21: string (nullable = true)
 |-- SUP_LABEL: string (nullable = true)
 |-- ShapeSTArea: 

In [4]:
from pyspark.sql import SparkSession
from sedona.spark import *
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import col

# Επεξεργασία Police Stations (X, Y)
# Διαβάζουμε το αρχείο
stations_df = spark.read.option("header", "true").csv("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Police_Stations.csv")

# Μετατρέπουμε τα X, Y σε Double και δημιουργούμε το Point
stations_df = stations_df \
    .withColumn("X", col("X").cast(DoubleType())) \
    .withColumn("Y", col("Y").cast(DoubleType())) \
    .withColumn("station_geom", ST_Point("X", "Y"))

print("Police Stations Schema:")
stations_df.printSchema()

# Επεξεργασία Crime Data (LON, LAT)
# Διαβάζουμε και τα δύο dataset (2010-2019 και 2020-)
crime_data_path_1 = "s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Crime_Data/LA_Crime_Data_2010_2019.csv"
crime_data_path_2 = "s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Crime_Data/LA_Crime_Data_2020_2025.csv"

# Φόρτωση
crimes_raw_1 = spark.read.option("header", "true").csv(crime_data_path_1)
crimes_raw_2 = spark.read.option("header", "true").csv(crime_data_path_2)

# Ένωση (Union) των δύο datasets
crimes_df = crimes_raw_1.union(crimes_raw_2)

# Μετατροπή σε Double και Φιλτράρισμα Null Island (0,0)
crimes_df = crimes_df \
    .withColumn("LAT", col("LAT").cast(DoubleType())) \
    .withColumn("LON", col("LON").cast(DoubleType())) \
    .filter((col("LAT") != 0.0) & (col("LON") != 0.0) & col("LAT").isNotNull() & col("LON").isNotNull())

# Δημιουργία του Point για τα εγκλήματα
crimes_df = crimes_df.withColumn("crime_geom", ST_Point("LON", "LAT"))

print("Crime Data Schema with Geometry:")
crimes_df.select("LAT", "LON", "crime_geom").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Police Stations Schema:
root
 |-- X: double (nullable = true)
 |-- Y: double (nullable = true)
 |-- FID: string (nullable = true)
 |-- DIVISION: string (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- PREC: string (nullable = true)
 |-- station_geom: geometry (nullable = true)

Crime Data Schema with Geometry:
+-------+---------+--------------------+
|    LAT|      LON|          crime_geom|
+-------+---------+--------------------+
|33.9825|-118.2695|POINT (-118.2695 ...|
|33.9599|-118.3962|POINT (-118.3962 ...|
|34.0224|-118.2524|POINT (-118.2524 ...|
|34.1016|-118.3295|POINT (-118.3295 ...|
|34.0387|-118.2488|POINT (-118.2488 ...|
+-------+---------+--------------------+
only showing top 5 rows

In [5]:
from pyspark.sql import Window
from pyspark.sql.functions import broadcast, col, row_number, count, avg, desc
import time

# Cross Join με χρήση Broadcast
joined_df = crimes_df.crossJoin(broadcast(stations_df))

# Υπολογισμός Απόστασης (σε χιλιόμετρα)
dist_df = joined_df.withColumn(
    "distance_km", 
    ST_DistanceSphere("crime_geom", "station_geom") / 1000
)

# Εύρεση του Πλησιέστερου Τμήματος (Window Function)
window_spec = Window.partitionBy("DR_NO").orderBy(col("distance_km").asc())

nearest_crime_station_df = dist_df \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .drop("rank") # Καθαρισμός

# Aggregations ανά Τμήμα
result_df = nearest_crime_station_df \
    .groupBy("DIVISION") \
    .agg(
        avg("distance_km").alias("average_distance"),
        count("*").alias("crime_count")
    ) \
    .orderBy(col("crime_count").desc())

start_time = time.time()
# Εμφάνιση αποτελεσμάτων
result_df.show()
end_time = time.time()

print(f"Total execution time: {end_time - start_time} seconds")
# Εμφάνιση πλάνου
result_df.explain()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------+------------------+-----------+
|        DIVISION|  average_distance|crime_count|
+----------------+------------------+-----------+
|       HOLLYWOOD| 2.073575473968209|     224124|
|        VAN NUYS| 2.939256360612646|     208129|
|       SOUTHWEST|2.1914685251016546|     189119|
|        WILSHIRE|2.5932983742728704|     186383|
|     77TH STREET|1.7171149163340818|     170620|
| NORTH HOLLYWOOD|2.6424992075261544|     168096|
|         OLYMPIC|1.7289319104161371|     162805|
|         PACIFIC|3.8534974118984215|     162027|
|         CENTRAL|0.9933242673630879|     154689|
|         RAMPART|1.5342201910926923|     153204|
|       SOUTHEAST| 2.443914918878559|     143803|
|     WEST VALLEY|3.0215716977222846|     136622|
|        FOOTHILL| 4.260099759728346|     132482|
|         TOPANGA|  3.29698920989731|     131054|
|          HARBOR|3.7017170626322615|     127071|
|      HOLLENBECK| 2.677452486073298|     116235|
|WEST LOS ANGELES| 2.789521497554683|     115969|


In [6]:
%%configure -f
{
    "conf":{
        "spark.executor.instances": "2",
        "spark.executor.memory": "4g",
        "spark.executor.cores": "2"
    }
}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1447,application_1765289937462_1435,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1429,application_1765289937462_1417,pyspark,idle,Link,Link,None,
1436,application_1765289937462_1424,pyspark,idle,Link,Link,None,
1437,application_1765289937462_1425,pyspark,idle,Link,Link,None,
1438,application_1765289937462_1426,pyspark,idle,Link,Link,None,
1441,application_1765289937462_1429,pyspark,idle,Link,Link,None,
1446,application_1765289937462_1434,pyspark,idle,Link,Link,None,
1447,application_1765289937462_1435,pyspark,idle,Link,Link,None,✔


In [7]:
from sedona.spark import *
from pyspark.sql.functions import col
from pyspark.sql import SparkSession

# Create spark Session
spark = SparkSession.builder \
    .appName("GeoJSON read") \
    .getOrCreate()

# Create sedona context
sedona = SedonaContext.create(spark)
# Read the file from s3
geojson_path = "s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Census_Blocks_2020.geojson"
blocks_df = sedona.read.format("geojson") \
            .option("multiLine", "true").load(geojson_path) \
            .selectExpr("explode(features) as features") \
            .select("features.*")
# Formatting magic
flattened_df = blocks_df.select( \
                [col(f"properties.{col_name}").alias(col_name) for col_name in \
                blocks_df.schema["properties"].dataType.fieldNames()] + ["geometry"]) \
            .drop("properties") \
            .drop("type")
# Print schema
flattened_df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- BG20: string (nullable = true)
 |-- BG20FIP_CURRENT: string (nullable = true)
 |-- BGFIP20: string (nullable = true)
 |-- CB20: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- CITYCOMM: string (nullable = true)
 |-- CITYCOMM_CURRENT: string (nullable = true)
 |-- CITY_CURRENT: string (nullable = true)
 |-- COMM: string (nullable = true)
 |-- COMM_CURRENT: string (nullable = true)
 |-- COUNTY: string (nullable = true)
 |-- CT20: string (nullable = true)
 |-- CTCB20: string (nullable = true)
 |-- FEAT_TYPE: string (nullable = true)
 |-- FIP20: string (nullable = true)
 |-- FIP_CURRENT: string (nullable = true)
 |-- HD22: long (nullable = true)
 |-- HD_NAME: string (nullable = true)
 |-- HOUSING20: long (nullable = true)
 |-- OBJECTID: long (nullable = true)
 |-- POP20: long (nullable = true)
 |-- SPA22: long (nullable = true)
 |-- SPA_NAME: string (nullable = true)
 |-- SUP21: string (nullable = true)
 |-- SUP_LABEL: string (nullable = true)
 |-- ShapeSTArea: 

In [8]:
from pyspark.sql import SparkSession
from sedona.spark import *
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import col

# Επεξεργασία Police Stations (X, Y)
# Διαβάζουμε το αρχείο
stations_df = spark.read.option("header", "true").csv("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Police_Stations.csv")

# Μετατρέπουμε τα X, Y σε Double και δημιουργούμε το Point
stations_df = stations_df \
    .withColumn("X", col("X").cast(DoubleType())) \
    .withColumn("Y", col("Y").cast(DoubleType())) \
    .withColumn("station_geom", ST_Point("X", "Y"))

print("Police Stations Schema:")
stations_df.printSchema()

# Επεξεργασία Crime Data (LON, LAT)
# Διαβάζουμε και τα δύο dataset (2010-2019 και 2020-)
crime_data_path_1 = "s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Crime_Data/LA_Crime_Data_2010_2019.csv"
crime_data_path_2 = "s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Crime_Data/LA_Crime_Data_2020_2025.csv"

# Φόρτωση
crimes_raw_1 = spark.read.option("header", "true").csv(crime_data_path_1)
crimes_raw_2 = spark.read.option("header", "true").csv(crime_data_path_2)

# Ένωση (Union) των δύο datasets
crimes_df = crimes_raw_1.union(crimes_raw_2)

# Μετατροπή σε Double και Φιλτράρισμα Null Island (0,0)
crimes_df = crimes_df \
    .withColumn("LAT", col("LAT").cast(DoubleType())) \
    .withColumn("LON", col("LON").cast(DoubleType())) \
    .filter((col("LAT") != 0.0) & (col("LON") != 0.0) & col("LAT").isNotNull() & col("LON").isNotNull())

# Δημιουργία του Point για τα εγκλήματα
crimes_df = crimes_df.withColumn("crime_geom", ST_Point("LON", "LAT"))

print("Crime Data Schema with Geometry:")
crimes_df.select("LAT", "LON", "crime_geom").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Police Stations Schema:
root
 |-- X: double (nullable = true)
 |-- Y: double (nullable = true)
 |-- FID: string (nullable = true)
 |-- DIVISION: string (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- PREC: string (nullable = true)
 |-- station_geom: geometry (nullable = true)

Crime Data Schema with Geometry:
+-------+---------+--------------------+
|    LAT|      LON|          crime_geom|
+-------+---------+--------------------+
|33.9825|-118.2695|POINT (-118.2695 ...|
|33.9599|-118.3962|POINT (-118.3962 ...|
|34.0224|-118.2524|POINT (-118.2524 ...|
|34.1016|-118.3295|POINT (-118.3295 ...|
|34.0387|-118.2488|POINT (-118.2488 ...|
+-------+---------+--------------------+
only showing top 5 rows

In [9]:
from pyspark.sql import Window
from pyspark.sql.functions import broadcast, col, row_number, count, avg, desc
import time

# Cross Join με χρήση Broadcast
joined_df = crimes_df.crossJoin(broadcast(stations_df))

# Υπολογισμός Απόστασης (σε χιλιόμετρα)
dist_df = joined_df.withColumn(
    "distance_km", 
    ST_DistanceSphere("crime_geom", "station_geom") / 1000
)

# Εύρεση του Πλησιέστερου Τμήματος (Window Function)
window_spec = Window.partitionBy("DR_NO").orderBy(col("distance_km").asc())

nearest_crime_station_df = dist_df \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .drop("rank") # Καθαρισμός

# Aggregations ανά Τμήμα
result_df = nearest_crime_station_df \
    .groupBy("DIVISION") \
    .agg(
        avg("distance_km").alias("average_distance"),
        count("*").alias("crime_count")
    ) \
    .orderBy(col("crime_count").desc())

start_time = time.time()
# Εμφάνιση αποτελεσμάτων
result_df.show()
end_time = time.time()

print(f"Total execution time: {end_time - start_time} seconds")
# Εμφάνιση πλάνου
result_df.explain()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------+------------------+-----------+
|        DIVISION|  average_distance|crime_count|
+----------------+------------------+-----------+
|       HOLLYWOOD| 2.073575473968208|     224124|
|        VAN NUYS| 2.939256360612646|     208129|
|       SOUTHWEST|2.1914685251016546|     189119|
|        WILSHIRE|2.5932983742728704|     186383|
|     77TH STREET|1.7171149163340818|     170620|
| NORTH HOLLYWOOD|2.6424992075261535|     168096|
|         OLYMPIC| 1.728931910416137|     162805|
|         PACIFIC|3.8534974118984233|     162027|
|         CENTRAL|0.9933242673630879|     154689|
|         RAMPART|1.5342201910926925|     153204|
|       SOUTHEAST| 2.443914918878558|     143803|
|     WEST VALLEY|3.0215716977222837|     136622|
|        FOOTHILL| 4.260099759728347|     132482|
|         TOPANGA|3.2969892098973097|     131054|
|          HARBOR|3.7017170626322624|     127071|
|      HOLLENBECK| 2.677452486073299|     116235|
|WEST LOS ANGELES|2.7895214975546816|     115969|


In [10]:
%%configure -f
{
    "conf":{
        "spark.executor.instances": "2",
        "spark.executor.memory": "8g",
        "spark.executor.cores": "4"
    }
}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1448,application_1765289937462_1436,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1429,application_1765289937462_1417,pyspark,idle,Link,Link,None,
1436,application_1765289937462_1424,pyspark,idle,Link,Link,None,
1437,application_1765289937462_1425,pyspark,idle,Link,Link,None,
1438,application_1765289937462_1426,pyspark,idle,Link,Link,None,
1441,application_1765289937462_1429,pyspark,idle,Link,Link,None,
1446,application_1765289937462_1434,pyspark,idle,Link,Link,None,
1448,application_1765289937462_1436,pyspark,idle,Link,Link,None,✔
1449,application_1765289937462_1437,pyspark,starting,Link,Link,None,


In [11]:
from sedona.spark import *
from pyspark.sql.functions import col
from pyspark.sql import SparkSession

# Create spark Session
spark = SparkSession.builder \
    .appName("GeoJSON read") \
    .getOrCreate()

# Create sedona context
sedona = SedonaContext.create(spark)
# Read the file from s3
geojson_path = "s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Census_Blocks_2020.geojson"
blocks_df = sedona.read.format("geojson") \
            .option("multiLine", "true").load(geojson_path) \
            .selectExpr("explode(features) as features") \
            .select("features.*")
# Formatting magic
flattened_df = blocks_df.select( \
                [col(f"properties.{col_name}").alias(col_name) for col_name in \
                blocks_df.schema["properties"].dataType.fieldNames()] + ["geometry"]) \
            .drop("properties") \
            .drop("type")
# Print schema
flattened_df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- BG20: string (nullable = true)
 |-- BG20FIP_CURRENT: string (nullable = true)
 |-- BGFIP20: string (nullable = true)
 |-- CB20: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- CITYCOMM: string (nullable = true)
 |-- CITYCOMM_CURRENT: string (nullable = true)
 |-- CITY_CURRENT: string (nullable = true)
 |-- COMM: string (nullable = true)
 |-- COMM_CURRENT: string (nullable = true)
 |-- COUNTY: string (nullable = true)
 |-- CT20: string (nullable = true)
 |-- CTCB20: string (nullable = true)
 |-- FEAT_TYPE: string (nullable = true)
 |-- FIP20: string (nullable = true)
 |-- FIP_CURRENT: string (nullable = true)
 |-- HD22: long (nullable = true)
 |-- HD_NAME: string (nullable = true)
 |-- HOUSING20: long (nullable = true)
 |-- OBJECTID: long (nullable = true)
 |-- POP20: long (nullable = true)
 |-- SPA22: long (nullable = true)
 |-- SPA_NAME: string (nullable = true)
 |-- SUP21: string (nullable = true)
 |-- SUP_LABEL: string (nullable = true)
 |-- ShapeSTArea: 

In [12]:
from pyspark.sql import SparkSession
from sedona.spark import *
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import col

# Επεξεργασία Police Stations (X, Y)
# Διαβάζουμε το αρχείο
stations_df = spark.read.option("header", "true").csv("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Police_Stations.csv")

# Μετατρέπουμε τα X, Y σε Double και δημιουργούμε το Point
stations_df = stations_df \
    .withColumn("X", col("X").cast(DoubleType())) \
    .withColumn("Y", col("Y").cast(DoubleType())) \
    .withColumn("station_geom", ST_Point("X", "Y"))

print("Police Stations Schema:")
stations_df.printSchema()

# Επεξεργασία Crime Data (LON, LAT)
# Διαβάζουμε και τα δύο dataset (2010-2019 και 2020-)
crime_data_path_1 = "s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Crime_Data/LA_Crime_Data_2010_2019.csv"
crime_data_path_2 = "s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Crime_Data/LA_Crime_Data_2020_2025.csv"

# Φόρτωση
crimes_raw_1 = spark.read.option("header", "true").csv(crime_data_path_1)
crimes_raw_2 = spark.read.option("header", "true").csv(crime_data_path_2)

# Ένωση (Union) των δύο datasets
crimes_df = crimes_raw_1.union(crimes_raw_2)

# Μετατροπή σε Double και Φιλτράρισμα Null Island (0,0)
crimes_df = crimes_df \
    .withColumn("LAT", col("LAT").cast(DoubleType())) \
    .withColumn("LON", col("LON").cast(DoubleType())) \
    .filter((col("LAT") != 0.0) & (col("LON") != 0.0) & col("LAT").isNotNull() & col("LON").isNotNull())

# Δημιουργία του Point για τα εγκλήματα
crimes_df = crimes_df.withColumn("crime_geom", ST_Point("LON", "LAT"))

print("Crime Data Schema with Geometry:")
crimes_df.select("LAT", "LON", "crime_geom").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Police Stations Schema:
root
 |-- X: double (nullable = true)
 |-- Y: double (nullable = true)
 |-- FID: string (nullable = true)
 |-- DIVISION: string (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- PREC: string (nullable = true)
 |-- station_geom: geometry (nullable = true)

Crime Data Schema with Geometry:
+-------+---------+--------------------+
|    LAT|      LON|          crime_geom|
+-------+---------+--------------------+
|33.9825|-118.2695|POINT (-118.2695 ...|
|33.9599|-118.3962|POINT (-118.3962 ...|
|34.0224|-118.2524|POINT (-118.2524 ...|
|34.1016|-118.3295|POINT (-118.3295 ...|
|34.0387|-118.2488|POINT (-118.2488 ...|
+-------+---------+--------------------+
only showing top 5 rows

In [13]:
from pyspark.sql import Window
from pyspark.sql.functions import broadcast, col, row_number, count, avg, desc
import time

# Cross Join με χρήση Broadcast
joined_df = crimes_df.crossJoin(broadcast(stations_df))

# Υπολογισμός Απόστασης (σε χιλιόμετρα)
dist_df = joined_df.withColumn(
    "distance_km", 
    ST_DistanceSphere("crime_geom", "station_geom") / 1000
)

# Εύρεση του Πλησιέστερου Τμήματος (Window Function)
window_spec = Window.partitionBy("DR_NO").orderBy(col("distance_km").asc())

nearest_crime_station_df = dist_df \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .drop("rank") # Καθαρισμός

# Aggregations ανά Τμήμα
result_df = nearest_crime_station_df \
    .groupBy("DIVISION") \
    .agg(
        avg("distance_km").alias("average_distance"),
        count("*").alias("crime_count")
    ) \
    .orderBy(col("crime_count").desc())

start_time = time.time()
# Εμφάνιση αποτελεσμάτων
result_df.show()
end_time = time.time()

print(f"Total execution time: {end_time - start_time} seconds")
# Εμφάνιση πλάνου
result_df.explain()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------+------------------+-----------+
|        DIVISION|  average_distance|crime_count|
+----------------+------------------+-----------+
|       HOLLYWOOD| 2.073575473968208|     224124|
|        VAN NUYS| 2.939256360612646|     208129|
|       SOUTHWEST|2.1914685251016555|     189119|
|        WILSHIRE|2.5932983742728704|     186383|
|     77TH STREET|1.7171149163340818|     170620|
| NORTH HOLLYWOOD|2.6424992075261535|     168096|
|         OLYMPIC| 1.728931910416137|     162805|
|         PACIFIC|3.8534974118984215|     162027|
|         CENTRAL|0.9933242673630875|     154689|
|         RAMPART|1.5342201910926923|     153204|
|       SOUTHEAST|2.4439149188785585|     143803|
|     WEST VALLEY|3.0215716977222846|     136622|
|        FOOTHILL| 4.260099759728346|     132482|
|         TOPANGA| 3.296989209897311|     131054|
|          HARBOR|3.7017170626322624|     127071|
|      HOLLENBECK|2.6774524860732987|     116235|
|WEST LOS ANGELES|2.7895214975546816|     115969|
